In [ ]:
import pandas as pd
import numpy as np
import re
import os
from collections import defaultdict


class SpellingCorrector:

    def __init__(self, file_path):
        self.file_path = file_path
        self.error_to_correct = defaultdict(list)
        self.correct_words = set()

        self.load_data()

    def load_data(self):

        current_word = None

        with open(self.file_path, "r", encoding="utf-8") as file:

            for line in file:

                word = line.strip()

                if not word:
                    continue

                if word.startswith("$"):

                    current_word = word[1:].lower()
                    self.correct_words.add(current_word)

                elif current_word:

                    misspelling = word.lower()

                    self.error_to_correct[misspelling].append(
                        current_word
                    )

        print("\nDataset loaded successfully!")
        print("Correct words:", len(self.correct_words))
        print("Misspellings:", len(self.error_to_correct))

    def clean_text(self, word):

        word = word.lower().strip()

        word = re.sub(r"[^a-z_]", "", word)

        return word

    def levenshtein_distance(self, word1, word2):

        rows = len(word1) + 1
        cols = len(word2) + 1

        matrix = np.zeros((rows, cols), dtype=int)

        for i in range(rows):
            matrix[i][0] = i

        for j in range(cols):
            matrix[0][j] = j

        for i in range(1, rows):

            for j in range(1, cols):

                if word1[i - 1] == word2[j - 1]:
                    cost = 0
                else:
                    cost = 1

                matrix[i][j] = min(
                    matrix[i - 1][j] + 1,
                    matrix[i][j - 1] + 1,
                    matrix[i - 1][j - 1] + cost
                )

        return matrix[-1][-1]

    def correct(self, word, top_n=5):

        word = self.clean_text(word)

        if not word:
            return []

        if word in self.error_to_correct:

            return self.error_to_correct[word][:top_n]

        if word in self.correct_words:

            return [word]

        candidates = []

        for correct_word in self.correct_words:

            distance = self.levenshtein_distance(
                word,
                correct_word
            )

            candidates.append(
                (correct_word, distance)
            )

        candidates.sort(key=lambda x: x[1])

        return [
            word
            for word, distance in candidates[:top_n]
        ]

    def autocomplete(self, prefix, top_n=10):

        prefix = self.clean_text(prefix)

        if not prefix:
            return []

        matches = [
            word
            for word in self.correct_words
            if word.startswith(prefix)
        ]

        matches.sort()

        return matches[:top_n]


def find_dat_file():

    folder = r"D:\Jupyter\NLP Skills\Hometasks\Spelling Corrector\birkbeck.dat.txt"

    if not os.path.exists(folder):

        print("Folder not found:")
        print(folder)

        return None

    dat_files = [
        file
        for file in os.listdir(folder)
        if file.lower().endswith(".dat")
    ]

    if not dat_files:

        print("\nNo .dat file found in:")
        print(folder)

        print("\nPlease save the Birkbeck data as:")
        print("birkbeck.dat")

        return None

    print("\nDAT files found:")

    for i, file in enumerate(dat_files, 1):
        print(f"{i}. {file}")

    file_path = os.path.join(
        folder,
        dat_files[0]
    )

    print("\nUsing dataset:")
    print(file_path)

    return file_path

def main():

    file_path = r"D:\Jupyter\NLP Skills\Hometasks\Spelling Corrector\birkbeck.dat.txt"

    if not os.path.isfile(file_path):
        print("Dataset file not found:")
        print(file_path)
        return

    print("\nUsing dataset:")
    print(file_path)

    corrector = SpellingCorrector(file_path)

    print("\n===================================")
    print("       SPELLING CORRECTOR")
    print("===================================")

    while True:

        print("\n1. Spelling Correction")
        print("2. Autocomplete")
        print("3. Exit")

        choice = input("\nEnter choice: ")

        if choice == "1":

            word = input("\nEnter word: ")

            suggestions = corrector.correct(word)

            if not suggestions:
                print("\nNo suggestions found.")

            else:
                print("\nSuggestions:")

                for i, suggestion in enumerate(suggestions, 1):
                    print(f"{i}. {suggestion}")

        elif choice == "2":

            prefix = input("\nEnter prefix: ")

            suggestions = corrector.autocomplete(prefix)

            if not suggestions:
                print("\nNo autocomplete suggestions.")

            else:
                print("\nAutocomplete suggestions:")

                for i, suggestion in enumerate(suggestions, 1):
                    print(f"{i}. {suggestion}")

        elif choice == "3":

            print("\nProgram terminated.")
            break

        else:

            print("\nInvalid choice. Please enter 1, 2 or 3.")


if __name__ == "__main__":
    main()


Using dataset:
D:\Jupyter\NLP Skills\Hometasks\Spelling Corrector\birkbeck.dat.txt

Dataset loaded successfully!
Correct words: 6130
Misspellings: 33968

       SPELLING CORRECTOR

1. Spelling Correction
2. Autocomplete
3. Exit



Enter choice:  $abstract



Invalid choice. Please enter 1, 2 or 3.

1. Spelling Correction
2. Autocomplete
3. Exit



Enter choice:  1

Enter word:  asssassin



Suggestions:
1. assessing
2. assassinate
3. possession
4. accessing
5. satin

1. Spelling Correction
2. Autocomplete
3. Exit



Enter choice:  2

Enter prefix:  passs



No autocomplete suggestions.

1. Spelling Correction
2. Autocomplete
3. Exit



Enter choice:  ass



Invalid choice. Please enter 1, 2 or 3.

1. Spelling Correction
2. Autocomplete
3. Exit
